# NARCISSUS recreation — Colab driver

Run this notebook on Colab (browser **or** the VS Code Google Colab extension) connected to a GPU runtime. Cells progress from sanity-checks (cheap) to attack validation (medium) to the full grid (expensive — added in a later cell once the cheaper steps pass).

**Requires:** Colab Pro, Google account with Drive access.

## 1. Mount Drive

Drive holds the dataset cache, results CSV, and any trigger checkpoints — so a session crash doesn't lose work.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

WORK = '/content/drive/MyDrive/narcissus-recreation'
!mkdir -p {WORK}/data {WORK}/results {WORK}/checkpoints {WORK}/triggers
!ls {WORK}

## 2. Clone (or pull) the repo

In [ ]:
%cd /content
import os
if not os.path.isdir('narcissus-recreation-proposal'):
    !git clone https://github.com/alexzkhan07/narcissus-recreation-proposal.git
else:
    %cd narcissus-recreation-proposal
    !git pull
    %cd /content
%cd narcissus-recreation-proposal
!git log --oneline -3

## 3. GPU + deps check

In [ ]:
!nvidia-smi | head -20
import torch, torchvision, pandas, matplotlib
print('torch       :', torch.__version__, 'cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
print('torchvision :', torchvision.__version__)
print('pandas      :', pandas.__version__)
print('matplotlib  :', matplotlib.__version__)

## 4. Smoke test — 5 epochs, no-op trigger

Trains ResNet-18 on clean CIFAR-10 for 5 epochs, then evaluates `tar_acc` (clean accuracy on target class) and `asr` (with no trigger applied — should be near random ~10%).

If Tar-ACC is reasonable and ASR is in the 5–15% range, the pipeline is sound.

In [ ]:
!python3 code/train_eval.py --smoke --root {WORK}/data

## 5. Attack validation — BadNets, Blend, Label-Consistent, NARCISSUS at 50% target-class poison ratio

Trains four ResNet-18s for 30 epochs each. The arbitrary-trigger baselines should improve at this high target-class poison ratio, while NARCISSUS should be near the top because its trigger is feature-optimized.

30 epochs gets to ~88-90% clean accuracy — enough to read direction without spending the full 100-epoch budget. Total on A100: ~15-20 min.

In [ ]:
import sys, importlib
sys.path.insert(0, 'code')
for m in ('train_eval', 'attacks.badnets', 'attacks.blend', 'attacks.label_consistent', 'attacks.narcissus'):
    if m in sys.modules:
        importlib.reload(sys.modules[m])

from train_eval import train_and_eval, TrainConfig
from attacks.badnets import make_trigger_fn as make_badnets
from attacks.blend import make_trigger_fn as make_blend
from attacks.label_consistent import make_trigger_fns as make_lc
from attacks.narcissus import make_trigger_fn as make_narcissus

cfg = TrainConfig(epochs=30, batch_size=128, lr=0.1)
TARGET = 2  # 'bird' — matches the bundled NARCISSUS trigger
RATIO = 0.5
SEED = 0
ROOT = f'{WORK}/data'
lc_poison, lc_eval = make_lc()

results = []
for name, poison_trig, eval_trig in [
    ('BadNets',          make_badnets(patch_size=3, location='bottom-right', color=1.0), None),
    ('Blend',            make_blend(alpha=0.2, pattern_seed=1234), None),
    ('Label-Consistent', lc_poison, lc_eval),
    ('NARCISSUS',        make_narcissus(), None),
]:
    print(f'\n=== {name} (target={TARGET}, ratio={RATIO}, seed={SEED}, epochs={cfg.epochs}) ===')
    tar, asr = train_and_eval(
        trigger_fn=poison_trig, eval_trigger_fn=eval_trig,
        target_class=TARGET, target_class_poison_ratio=RATIO,
        seed=SEED, root=ROOT, cfg=cfg,
    )
    print(f'  -> tar_acc={tar*100:.2f}%  asr={asr*100:.2f}%')
    results.append((name, tar, asr))

print('\nsummary:')
for name, tar, asr in results:
    print(f'  {name:16s}  tar_acc={tar*100:5.2f}%  asr={asr*100:5.2f}%')

## 6. Run the Figure 3 sweep

This writes one row per `(method, target-class poison ratio, seed)` to Drive and skips rows that already exist, so it can be interrupted and resumed. The ratios match the paper's Figure 3 x-axis, including the low 0.5% point where NARCISSUS separates from the arbitrary-trigger baselines.

In [ ]:
RESULTS_CSV = f'{WORK}/results/figure3_results.csv'
FULL_EPOCHS = 100
RATIOS = '0,0.5,5,10,20,30,70'
SEEDS = '0,1,2'
METHODS = 'BadNets,Blend,Label-Consistent,Ours'

!python3 code/run_grid.py \
    --root {WORK}/data \
    --out {RESULTS_CSV} \
    --epochs {FULL_EPOCHS} \
    --target-class 2 \
    --ratios {RATIOS} \
    --seeds {SEEDS} \
    --methods {METHODS}

## 7. Render Figure 3

The plotted image is also written to Drive beside the CSV.

In [ ]:
FIGURE_OUT = f'{WORK}/results/figure3.png'
!python3 code/plot_figure3.py --csv {RESULTS_CSV} --out {FIGURE_OUT}
from IPython.display import Image
Image(FIGURE_OUT)